# Notebook 10 — Walk-Forward Validation & Model Selection

This notebook performs the next critical stage after the initial statistical, machine-learning, and deep-learning baselines: **leakage-controlled walk-forward validation**.

The objective is to determine whether candidate next-day direction models remain useful across multiple historical periods rather than only one final holdout.

### Candidate models

- Logistic Regression
- Random Forest
- HistGradientBoosting
- XGBoost when available

### Validation design

An expanding-window walk-forward procedure is used:

1. Train on an initial historical window.
2. Predict the next validation block.
3. Expand the training window.
4. Repeat until the historical sample is exhausted.

The test observations of each fold are never used to fit that fold's model.

This notebook focuses on classical ML models because repeatedly retraining LSTM/GRU models over many folds would be unnecessarily expensive for the first walk-forward selection stage. Deep-learning models remain available from Notebook 09 for later focused comparison.


## 1. Imports

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    xgb_available = True
    print("XGBoost available.")
except ImportError:
    XGBClassifier = None
    xgb_available = False
    print("XGBoost unavailable; continuing without it.")

print("Imports loaded successfully.")


## 2. Configuration and Paths

In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").exists():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data/quant-trading-research"),
    ]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break

MASTER_PATH = PROJECT_ROOT / "data" / "raw" / "sp500_1950_present.csv"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
REPORT_DIR = PROJECT_ROOT / "reports" / "generated"

for path in [INTERIM_DIR, FIGURE_DIR, TABLE_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

EXPECTED_COLUMNS = [
    "Date", "Open", "High", "Low", "Close", "Adj.Close", "Volume"
]

# Walk-forward configuration
INITIAL_TRAIN_FRACTION = 0.60
VALIDATION_BLOCK_FRACTION = 0.05
MIN_TRAIN_ROWS = 1000

# Probability threshold used for the standard directional signal
DEFAULT_THRESHOLD = 0.50

print(f"Master dataset: {MASTER_PATH}")


## 3. Load and Validate the Master Dataset

In [ ]:
if not MASTER_PATH.exists():
    raise FileNotFoundError(
        f"Master dataset not found: {MASTER_PATH}. "
        "Run the earlier notebooks first."
    )

df = pd.read_csv(
    MASTER_PATH,
    low_memory=False
)

if list(df.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        f"Unexpected schema. Expected {EXPECTED_COLUMNS}; "
        f"received {list(df.columns)}"
    )

df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

for column in EXPECTED_COLUMNS[1:]:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

df = (
    df.sort_values("Date")
    .reset_index(drop=True)
)

if df["Date"].isna().any():
    raise ValueError("Invalid dates detected.")

if df["Date"].duplicated().any():
    raise ValueError("Duplicate dates detected.")

if df[
    ["Open", "High", "Low", "Close", "Adj.Close", "Volume"]
].isna().any().any():
    raise ValueError("Missing OHLCV values detected.")

print(f"Rows: {len(df):,}")
print(
    f"Date range: {df['Date'].min().date()} "
    f"→ {df['Date'].max().date()}"
)


## 4. Build the Supervised Learning Dataset

All predictors are constructed from observations available at or before the prediction date.

The target is the next trading day's return direction.


In [ ]:
df["return_1d"] = df["Close"].pct_change()
df["return_5d"] = df["Close"].pct_change(5)
df["return_21d"] = df["Close"].pct_change(21)
df["return_63d"] = df["Close"].pct_change(63)
df["return_126d"] = df["Close"].pct_change(126)
df["return_252d"] = df["Close"].pct_change(252)

df["volatility_5d"] = (
    df["return_1d"].rolling(5).std()
    * np.sqrt(252)
)

df["volatility_21d"] = (
    df["return_1d"].rolling(21).std()
    * np.sqrt(252)
)

df["volatility_63d"] = (
    df["return_1d"].rolling(63).std()
    * np.sqrt(252)
)

df["sma_20"] = df["Close"].rolling(20).mean()
df["sma_50"] = df["Close"].rolling(50).mean()
df["sma_200"] = df["Close"].rolling(200).mean()

df["price_to_sma_20"] = (
    df["Close"] / df["sma_20"] - 1
)

df["price_to_sma_50"] = (
    df["Close"] / df["sma_50"] - 1
)

df["price_to_sma_200"] = (
    df["Close"] / df["sma_200"] - 1
)

df["sma_50_vs_sma_200"] = (
    df["sma_50"] / df["sma_200"] - 1
)

df["range_pct"] = (
    (df["High"] - df["Low"]) / df["Close"]
)

df["intraday_return"] = (
    df["Close"] / df["Open"] - 1
)

df["overnight_return"] = (
    df["Open"] / df["Close"].shift(1) - 1
)

df["volume_ratio_20"] = (
    df["Volume"] /
    df["Volume"].rolling(20).mean()
)

df["volume_ratio_63"] = (
    df["Volume"] /
    df["Volume"].rolling(63).mean()
)

df["return_1d_lag1"] = df["return_1d"].shift(1)
df["return_1d_lag2"] = df["return_1d"].shift(2)
df["return_1d_lag3"] = df["return_1d"].shift(3)
df["return_1d_lag5"] = df["return_1d"].shift(5)
df["return_1d_lag10"] = df["return_1d"].shift(10)

df["volatility_21d_lag1"] = (
    df["volatility_21d"].shift(1)
)

df["range_pct_lag1"] = (
    df["range_pct"].shift(1)
)

df["volume_ratio_20_lag1"] = (
    df["volume_ratio_20"].shift(1)
)

df["next_day_return"] = (
    df["Close"].shift(-1) / df["Close"] - 1
)

df["target"] = (
    df["next_day_return"] > 0
).astype(int)


## 5. Define the Feature Set

In [ ]:
feature_columns = [
    "return_1d",
    "return_5d",
    "return_21d",
    "return_63d",
    "return_126d",
    "return_252d",
    "volatility_5d",
    "volatility_21d",
    "volatility_63d",
    "price_to_sma_20",
    "price_to_sma_50",
    "price_to_sma_200",
    "sma_50_vs_sma_200",
    "range_pct",
    "intraday_return",
    "overnight_return",
    "volume_ratio_20",
    "volume_ratio_63",
    "return_1d_lag1",
    "return_1d_lag2",
    "return_1d_lag3",
    "return_1d_lag5",
    "return_1d_lag10",
    "volatility_21d_lag1",
    "range_pct_lag1",
    "volume_ratio_20_lag1",
]

model_df = df[
    [
        "Date",
        "Close",
        "next_day_return",
        "target",
    ] + feature_columns
].dropna(
    subset=feature_columns + [
        "next_day_return",
        "target",
    ]
).reset_index(drop=True)

print(f"Modeling rows: {len(model_df):,}")
print(f"Features: {len(feature_columns)}")


## 6. Walk-Forward Fold Construction

The folds are generated chronologically.

Example:

```text
Fold 1: [Train ----------------] [Validation]
Fold 2: [Train ------------------------] [Validation]
Fold 3: [Train --------------------------------] [Validation]
...
```

The training set expands after every validation block.


In [ ]:
n_rows = len(model_df)

initial_train_size = max(
    MIN_TRAIN_ROWS,
    int(n_rows * INITIAL_TRAIN_FRACTION)
)

validation_block_size = max(
    100,
    int(n_rows * VALIDATION_BLOCK_FRACTION)
)

folds = []

train_end = initial_train_size
fold_id = 1

while train_end < n_rows:
    validation_start = train_end
    validation_end = min(
        train_end + validation_block_size,
        n_rows
    )

    if validation_end <= validation_start:
        break

    folds.append({
        "fold": fold_id,
        "train_start": 0,
        "train_end": train_end,
        "validation_start": validation_start,
        "validation_end": validation_end,
    })

    train_end = validation_end
    fold_id += 1

fold_table = pd.DataFrame(folds)

fold_table["train_start_date"] = (
    fold_table["train_start"]
    .map(model_df["Date"])
)

fold_table["train_end_date"] = (
    (fold_table["train_end"] - 1)
    .map(model_df["Date"])
)

fold_table["validation_start_date"] = (
    fold_table["validation_start"]
    .map(model_df["Date"])
)

fold_table["validation_end_date"] = (
    (fold_table["validation_end"] - 1)
    .map(model_df["Date"])
)

display(fold_table)

fold_table.to_csv(
    TABLE_DIR / "sp500_walk_forward_folds.csv",
    index=False
)


## 7. Visualize Walk-Forward Structure

In [ ]:
fig = plt.figure(figsize=(14, 8))

for row_number, row in fold_table.iterrows():
    y = row_number

    plt.plot(
        [
            row["train_start_date"],
            row["train_end_date"],
        ],
        [y, y],
        linewidth=8,
        label="Train" if row_number == 0 else None,
    )

    plt.plot(
        [
            row["validation_start_date"],
            row["validation_end_date"],
        ],
        [y, y],
        linewidth=8,
        label="Validation" if row_number == 0 else None,
    )

plt.yticks(
    range(len(fold_table)),
    [
        f"Fold {int(x)}"
        for x in fold_table["fold"]
    ],
)

plt.title("Expanding-Window Walk-Forward Validation")
plt.xlabel("Date")
plt.ylabel("Fold")
plt.legend()
plt.grid(True, axis="x", alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_walk_forward_validation_structure.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 8. Model Factory

In [ ]:
def create_models():
    models = {}

    models["Logistic Regression"] = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42,
            ),
        ),
    ])

    models["Random Forest"] = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=250,
                max_depth=8,
                min_samples_leaf=10,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ])

    models["HistGradientBoosting"] = Pipeline([
        (
            "model",
            HistGradientBoostingClassifier(
                max_iter=250,
                learning_rate=0.05,
                max_leaf_nodes=15,
                l2_regularization=1.0,
                random_state=42,
            ),
        ),
    ])

    if xgb_available:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.80,
            colsample_bytree=0.80,
            min_child_weight=5,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        )

    return models


print(
    "Candidate models:",
    list(create_models().keys())
)


## 9. Walk-Forward Evaluation Function

In [ ]:
def evaluate_fold(
    model_name,
    model,
    train_slice,
    validation_slice,
):
    X_train = train_slice[feature_columns]
    y_train = train_slice["target"]

    X_valid = validation_slice[feature_columns]
    y_valid = validation_slice["target"]

    model.fit(
        X_train,
        y_train
    )

    probabilities = model.predict_proba(
        X_valid
    )[:, 1]

    predictions = (
        probabilities >= DEFAULT_THRESHOLD
    ).astype(int)

    metrics = {
        "accuracy": accuracy_score(
            y_valid,
            predictions,
        ),
        "precision": precision_score(
            y_valid,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_valid,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_valid,
            predictions,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y_valid,
            probabilities,
        ),
    }

    result = {
        "model": model_name,
        "validation_start": validation_slice["Date"].min(),
        "validation_end": validation_slice["Date"].max(),
        "train_rows": len(train_slice),
        "validation_rows": len(validation_slice),
        **metrics,
    }

    prediction_frame = validation_slice[
        [
            "Date",
            "Close",
            "next_day_return",
            "target",
        ]
    ].copy()

    prediction_frame["model"] = model_name
    prediction_frame["probability_up"] = probabilities
    prediction_frame["prediction"] = predictions

    return result, prediction_frame


## 10. Execute Walk-Forward Validation

Every model is refitted independently inside every fold.

This is intentionally more computationally expensive than a single train/test split because it better approximates repeated model deployment through history.


In [ ]:
fold_results = []
fold_predictions = []

candidate_models = list(create_models().keys())

for model_name in candidate_models:
    print("=" * 72)
    print(f"MODEL: {model_name}")
    print("=" * 72)

    for _, fold in fold_table.iterrows():
        fold_number = int(fold["fold"])

        train_slice = model_df.iloc[
            int(fold["train_start"]):
            int(fold["train_end"])
        ].copy()

        validation_slice = model_df.iloc[
            int(fold["validation_start"]):
            int(fold["validation_end"])
        ].copy()

        models = create_models()
        model = models[model_name]

        result, prediction_frame = evaluate_fold(
            model_name,
            model,
            train_slice,
            validation_slice,
        )

        result["fold"] = fold_number

        fold_results.append(result)

        prediction_frame["fold"] = fold_number

        fold_predictions.append(
            prediction_frame
        )

        print(
            f"Fold {fold_number:02d} | "
            f"{result['validation_start'].date()} → "
            f"{result['validation_end'].date()} | "
            f"ROC-AUC={result['roc_auc']:.4f} | "
            f"F1={result['f1']:.4f}"
        )

fold_results_df = pd.DataFrame(
    fold_results
)

fold_predictions_df = pd.concat(
    fold_predictions,
    ignore_index=True
)

print(
    f"Completed {len(fold_results_df):,} "
    "model-fold evaluations."
)


## 11. Save Fold-Level Results

In [ ]:
fold_results_df = fold_results_df.sort_values(
    ["model", "fold"]
).reset_index(drop=True)

fold_results_df.to_csv(
    TABLE_DIR / "sp500_walk_forward_fold_results.csv",
    index=False
)

fold_predictions_path = (
    INTERIM_DIR /
    "sp500_walk_forward_predictions.parquet"
)

fold_predictions_df.to_parquet(
    fold_predictions_path,
    index=False
)

display(fold_results_df.head(20))

print(f"Saved: {fold_predictions_path}")


## 12. Aggregate Model Performance Across Folds

In [ ]:
aggregate_metrics = (
    fold_results_df
    .groupby("model")
    .agg(
        folds=("fold", "count"),
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        mean_precision=("precision", "mean"),
        std_precision=("precision", "std"),
        mean_recall=("recall", "mean"),
        std_recall=("recall", "std"),
        mean_f1=("f1", "mean"),
        std_f1=("f1", "std"),
        mean_roc_auc=("roc_auc", "mean"),
        std_roc_auc=("roc_auc", "std"),
    )
    .reset_index()
)

aggregate_metrics = aggregate_metrics.sort_values(
    [
        "mean_roc_auc",
        "mean_f1",
    ],
    ascending=False
).reset_index(drop=True)

display(aggregate_metrics)

aggregate_metrics.to_csv(
    TABLE_DIR / "sp500_walk_forward_model_comparison.csv",
    index=False
)


## 13. Model Stability Across Folds

In [ ]:
stability_table = (
    fold_results_df
    .groupby("model")
    .agg(
        roc_auc_min=("roc_auc", "min"),
        roc_auc_max=("roc_auc", "max"),
        roc_auc_median=("roc_auc", "median"),
        f1_min=("f1", "min"),
        f1_max=("f1", "max"),
        f1_median=("f1", "median"),
    )
    .reset_index()
)

display(stability_table)

stability_table.to_csv(
    TABLE_DIR / "sp500_walk_forward_model_stability.csv",
    index=False
)


## 14. Fold-by-Fold ROC-AUC

In [ ]:
fig = plt.figure(figsize=(14, 6))

for model_name in fold_results_df["model"].unique():
    subset = fold_results_df[
        fold_results_df["model"] == model_name
    ]

    plt.plot(
        subset["fold"],
        subset["roc_auc"],
        marker="o",
        label=model_name,
    )

plt.axhline(
    0.50,
    linestyle="--",
    label="Random-direction reference",
)

plt.title("Walk-Forward ROC-AUC by Fold")
plt.xlabel("Fold")
plt.ylabel("ROC-AUC")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_walk_forward_roc_auc_by_fold.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 15. Fold-by-Fold F1

In [ ]:
fig = plt.figure(figsize=(14, 6))

for model_name in fold_results_df["model"].unique():
    subset = fold_results_df[
        fold_results_df["model"] == model_name
    ]

    plt.plot(
        subset["fold"],
        subset["f1"],
        marker="o",
        label=model_name,
    )

plt.title("Walk-Forward F1 by Fold")
plt.xlabel("Fold")
plt.ylabel("F1")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_walk_forward_f1_by_fold.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 16. Cross-Fold Consistency Score

A useful research model should not rely on a single unusually strong fold.

The consistency score measures the fraction of folds where ROC-AUC exceeds 0.50.


In [ ]:
consistency = (
    fold_results_df
    .assign(
        roc_auc_above_random=lambda x:
            x["roc_auc"] > 0.50
    )
    .groupby("model")
    .agg(
        folds=("fold", "count"),
        folds_above_0_50=("roc_auc_above_random", "sum"),
    )
    .reset_index()
)

consistency["consistency_rate"] = (
    consistency["folds_above_0_50"]
    / consistency["folds"]
)

display(consistency)

consistency.to_csv(
    TABLE_DIR / "sp500_walk_forward_consistency.csv",
    index=False
)


## 17. Pooled Out-of-Sample Predictions

All validation predictions are concatenated chronologically.

These predictions represent historical pseudo-out-of-sample observations because each prediction comes from a model trained only on information available before that fold's validation period.


In [ ]:
pooled_predictions = (
    fold_predictions_df
    .sort_values(
        ["model", "Date"]
    )
    .reset_index(drop=True)
)

pooled_metrics = []

for model_name in pooled_predictions["model"].unique():
    subset = pooled_predictions[
        pooled_predictions["model"] == model_name
    ]

    pooled_metrics.append({
        "model": model_name,
        "observations": len(subset),
        "accuracy": accuracy_score(
            subset["target"],
            subset["prediction"],
        ),
        "precision": precision_score(
            subset["target"],
            subset["prediction"],
            zero_division=0,
        ),
        "recall": recall_score(
            subset["target"],
            subset["prediction"],
            zero_division=0,
        ),
        "f1": f1_score(
            subset["target"],
            subset["prediction"],
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            subset["target"],
            subset["probability_up"],
        ),
    })

pooled_metrics_df = pd.DataFrame(
    pooled_metrics
).sort_values(
    "roc_auc",
    ascending=False
)

display(pooled_metrics_df)

pooled_metrics_df.to_csv(
    TABLE_DIR / "sp500_walk_forward_pooled_metrics.csv",
    index=False
)


## 18. Probability Distribution of the Best Walk-Forward Model

In [ ]:
best_walk_forward_model = (
    aggregate_metrics.iloc[0]["model"]
)

best_predictions = pooled_predictions[
    pooled_predictions["model"] ==
    best_walk_forward_model
].copy()

fig = plt.figure(figsize=(12, 6))

plt.hist(
    best_predictions["probability_up"],
    bins=40,
)

plt.axvline(
    DEFAULT_THRESHOLD,
    linestyle="--",
    label="0.50 threshold",
)

plt.title(
    f"Walk-Forward Probability Distribution — "
    f"{best_walk_forward_model}"
)

plt.xlabel("Probability of Positive Next-Day Return")
plt.ylabel("Observations")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_walk_forward_probability_distribution.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 19. Threshold Analysis on Pooled Walk-Forward Predictions

Threshold analysis evaluates signal quality at increasingly selective probability levels.

This is an exploratory diagnostic. It is not used to optimize a final production threshold in this notebook.


In [ ]:
threshold_rows = []

for model_name in pooled_predictions["model"].unique():
    subset = pooled_predictions[
        pooled_predictions["model"] == model_name
    ]

    for threshold in [
        0.50,
        0.55,
        0.60,
        0.65,
        0.70,
    ]:
        signal = (
            subset["probability_up"] >= threshold
        ).astype(int)

        threshold_rows.append({
            "model": model_name,
            "threshold": threshold,
            "signal_rate": signal.mean(),
            "precision": precision_score(
                subset["target"],
                signal,
                zero_division=0,
            ),
            "recall": recall_score(
                subset["target"],
                signal,
                zero_division=0,
            ),
            "f1": f1_score(
                subset["target"],
                signal,
                zero_division=0,
            ),
        })

threshold_analysis = pd.DataFrame(
    threshold_rows
)

display(threshold_analysis)

threshold_analysis.to_csv(
    TABLE_DIR / "sp500_walk_forward_threshold_analysis.csv",
    index=False
)


## 20. Select a Candidate Model

Selection is based primarily on:

1. Mean walk-forward ROC-AUC
2. Mean walk-forward F1
3. Cross-fold consistency

The notebook does not select a model based on one isolated fold.


In [ ]:
selection_table = aggregate_metrics.merge(
    consistency[
        [
            "model",
            "consistency_rate",
        ]
    ],
    on="model",
    how="left",
)

selection_table = selection_table.sort_values(
    [
        "mean_roc_auc",
        "mean_f1",
        "consistency_rate",
    ],
    ascending=False
).reset_index(drop=True)

display(selection_table)

candidate_model = selection_table.iloc[0]["model"]

print(
    f"Candidate model selected for subsequent research: "
    f"{candidate_model}"
)

selection_table.to_csv(
    TABLE_DIR / "sp500_walk_forward_selection_table.csv",
    index=False
)


## 21. Final Candidate Model Fit on All Available Pre-Test Data

The selected model is refitted on the full modeling dataset for artifact creation.

This model is **not yet a deployment model**. It is the candidate selected from the research validation process and will be evaluated further with trading-aware backtesting.


In [ ]:
candidate_models = create_models()

candidate_model_instance = (
    candidate_models[candidate_model]
)

candidate_model_instance.fit(
    model_df[feature_columns],
    model_df["target"],
)

candidate_model_path = (
    INTERIM_DIR /
    "sp500_walk_forward_selected_model.pkl"
)

import joblib

joblib.dump(
    candidate_model_instance,
    candidate_model_path,
)

print(
    f"Saved candidate model: "
    f"{candidate_model_path}"
)


## 22. Candidate Model Artifact Metadata

In [ ]:
candidate_metadata = {
    "candidate_model": candidate_model,
    "feature_columns": feature_columns,
    "training_rows": int(len(model_df)),
    "training_start": model_df["Date"].min().strftime("%Y-%m-%d"),
    "training_end": model_df["Date"].max().strftime("%Y-%m-%d"),
    "walk_forward": {
        "initial_train_fraction": INITIAL_TRAIN_FRACTION,
        "validation_block_fraction": VALIDATION_BLOCK_FRACTION,
        "folds": int(len(fold_table)),
    },
    "selection_metrics": selection_table.iloc[0].to_dict(),
    "methodological_note": (
        "Candidate selection uses expanding-window walk-forward "
        "validation. The final candidate artifact is for subsequent "
        "research/backtesting and is not evidence of deployable alpha."
    ),
}

metadata_path = (
    INTERIM_DIR /
    "sp500_walk_forward_candidate_metadata.json"
)

metadata_path.write_text(
    json.dumps(
        candidate_metadata,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        candidate_metadata,
        indent=2,
        default=str,
    )
)


## 23. Walk-Forward Research Report

In [ ]:
walk_forward_report = {
    "dataset": {
        "rows": int(len(df)),
        "modeling_rows": int(len(model_df)),
        "start": df["Date"].min().strftime("%Y-%m-%d"),
        "end": df["Date"].max().strftime("%Y-%m-%d"),
    },
    "validation_design": {
        "initial_train_fraction": INITIAL_TRAIN_FRACTION,
        "validation_block_fraction": VALIDATION_BLOCK_FRACTION,
        "fold_count": int(len(fold_table)),
        "expanding_window": True,
        "random_shuffle": False,
    },
    "fold_metrics": fold_results_df.to_dict(
        orient="records"
    ),
    "aggregate_metrics": aggregate_metrics.to_dict(
        orient="records"
    ),
    "pooled_metrics": pooled_metrics_df.to_dict(
        orient="records"
    ),
    "consistency": consistency.to_dict(
        orient="records"
    ),
    "candidate_model": candidate_model,
    "candidate_artifact": str(candidate_model_path),
    "methodological_note": (
        "Walk-forward predictions are pseudo-out-of-sample observations. "
        "The selected candidate must still be subjected to trading-aware "
        "backtesting, transaction-cost analysis, robustness checks, and "
        "final untouched evaluation before deployment."
    ),
}

report_path = (
    REPORT_DIR /
    "sp500_walk_forward_validation_report.json"
)

report_path.write_text(
    json.dumps(
        walk_forward_report,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        walk_forward_report,
        indent=2,
        default=str,
    )
)

print(f"\nSaved: {report_path}")


## 24. Final Raw Dataset Integrity Check

In [ ]:
master_check = pd.read_csv(
    MASTER_PATH,
    low_memory=False,
)

assert list(master_check.columns) == EXPECTED_COLUMNS
assert len(master_check) == len(df)

master_dates = pd.to_datetime(
    master_check["Date"],
    errors="coerce",
)

assert master_dates.notna().all()
assert master_dates.is_unique
assert master_dates.is_monotonic_increasing

print(
    "Raw master dataset integrity after walk-forward validation: PASS"
)
print(
    f"Master rows: {len(master_check):,}"
)


# Notebook 10 Complete

Notebook 10 has completed leakage-controlled expanding-window model validation.

### Main outputs

- Walk-forward fold definitions
- Chronological validation structure
- Logistic Regression walk-forward results
- Random Forest walk-forward results
- HistGradientBoosting walk-forward results
- XGBoost walk-forward results when available
- Fold-level metrics
- Aggregate metrics
- Pooled pseudo-out-of-sample predictions
- Model stability
- Cross-fold consistency
- Probability threshold diagnostics
- Candidate model selection
- Candidate model artifact
- Walk-forward research report
- Raw master-data integrity verification

### Important methodological boundary

Notebook 10 identifies a candidate model using repeated historical pseudo-out-of-sample evaluation. This is still **not proof of a tradable edge**.

The next stage must convert predictions into explicit trading rules and evaluate:
- returns
- drawdown
- Sharpe ratio
- Sortino ratio
- turnover
- transaction costs
- slippage
- exposure
- benchmark comparison
- robustness

**Next notebook:** Notebook 11 — Strategy Backtesting.

Run Notebook 10 from top to bottom and verify the outputs before proceeding.
